# Chapter 7 - Lab 3: <font color='blue'>Conversational Multi-Agent with AutoGen</font>

**<font color='purple'>Goal</font>**:
Solve a financial-analysis task with AutoGen's **conversational style**: two agents — an Assistant that *writes Python code* and a CodeExecutor that *runs it* — exchange messages until the task is complete.

Unlike Labs 1 and 2, there are **no handoffs and no shared state**. The conversation itself is the coordination mechanism. The Assistant generates code that pulls live data via `yfinance`, the CodeExecutor runs it, and they iterate.

**<font color='purple'>Tech stack</font>**:

* **AutoGen** (`pyautogen`) — `AssistantAgent`, `UserProxyAgent`, `LocalCommandLineCodeExecutor`.
* **yfinance** — live market data (no API key needed).
* **OpenAI** `gpt-4o`.

## 1. Install packages

In [ ]:
%pip install -q 'pyautogen[code-execution]' yfinance pandas matplotlib python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## 3. Configure the model

AutoGen takes a `config_list` so you can rotate keys / models / providers. We use a single OpenAI entry. `cache_seed` enables result caching for reproducibility while iterating on this lab.

In [ ]:
config_list = [
    {'model': 'gpt-4o', 'api_key': os.environ['OPENAI_API_KEY']},
]

llm_config = {'config_list': config_list, 'cache_seed': 42}

## 4. Create the code executor

`LocalCommandLineCodeExecutor` runs generated Python in a temporary directory. We cap execution at 60 seconds — long enough for charts but not for runaway loops.

In [ ]:
import tempfile
from autogen.coding import LocalCommandLineCodeExecutor

work_dir = tempfile.mkdtemp()
executor = LocalCommandLineCodeExecutor(timeout=60, work_dir=work_dir)

## 5. Define the two agents

The **AssistantAgent** is the brains — it writes Python code that fetches data and produces analysis. The **UserProxyAgent** acts as the code executor: it does not call an LLM (`llm_config=False`) and terminates the conversation when the assistant emits `TERMINATE`.

In [ ]:
from autogen import AssistantAgent, UserProxyAgent

assistant = AssistantAgent(
    name='assistant',
    llm_config=llm_config,
    system_message=(
        'You are a financial analyst assistant. When asked to analyze a company, '
        'write Python code using yfinance to fetch financial data and compute key metrics. '
        'Present results in a clear, structured format. End with the word TERMINATE.'
    ),
)

code_executor = UserProxyAgent(
    name='code_executor',
    human_input_mode='NEVER',
    code_execution_config={'executor': executor},
    llm_config=False,
    is_termination_msg=lambda m: 'TERMINATE' in m.get('content', ''),
)

## 6. Run the conversation

`initiate_chat` kicks off the back-and-forth. The code_executor sends the user's prompt, the assistant replies with Python code, the executor runs it and sends results back, and they iterate up to `max_turns` times.

In [ ]:
query = (
    'What is the current price and key financial metrics of NVIDIA? '
    'Plot the stock price over the last 6 months.'
)

chat_result = code_executor.initiate_chat(
    assistant,
    message=query,
    max_turns=5,
)

## 7. Inspect the conversation and cost

In [ ]:
print(f'Summary:\n{chat_result.summary}\n')
print('--- chat_history ---')
for msg in chat_result.chat_history[:6]:
    print(f'[{msg.get("role", "?")}] {str(msg.get("content", ""))[:200]}\n')
print('--- cost ---')
print(chat_result.cost)

## 8. Results

Three architectures for the same job: pipeline (Lab 1), leader-follower (Lab 2), conversation (Lab 3). **What to notice about conversational agents:**

* **Code generation is the killer feature.** When the assistant can write and execute code,   it can pull arbitrary data, compute anything, and even draw charts. No tool catalogue needed.
* **Coordination is implicit.** No shared state, no handoffs — just two agents talking. Simple   to set up, harder to debug when something goes wrong mid-conversation.
* **Sandboxing matters.** Generated code runs locally. In production, replace   `LocalCommandLineCodeExecutor` with a sandboxed runtime.
* **Pick the style that fits the problem.** Sequential for audited workflows (insurance, KYC).   Leader-follower for routing-heavy front desks. Conversational for exploratory analysis.